# Exercise Set 2

### True or False

**Q1.** Suppose the solar forecast for the week totaled more energy than the total demand. In this case it's guaranteed that no energy will be bought from the grid.
- True
- False v

### Formualtion and Coding

In [1]:
%pip install gurobipy

import gurobipy as gp
from gurobipy import GRB

batteries = ["Battery0", "Battery1"]
time_periods = range(180)
capacity = {"Battery0": 60, "Battery1": 80}
p_loss = {"Battery0": 0.95, "Battery1": 0.9}
initial = {"Battery0": 20, "Battery1": 30}

m = gp.Model()

# flow_in = m.addVars(batteries, time_periods, name="flow_in")
# flow_out = m.addVars(batteries, time_periods, name="flow_out")
# grid = m.addVars(time_periods, name="grid")
# state = m.addVars(batteries, time_periods, name="state")
# gen = m.addVars(time_periods, name="gen")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 11.9 MB/s  0:00:01
Restricted license - for non-production use only - expires 2028-11-29


In [4]:
.7*140

98.0

Feel free to use the commented code above to copy and paste.

**Q2.** Write a line of code that sets the objective coefficients for the energy storage problem (minimizing grid electricity purchased) using the `addVars()` command. Check your code by writing to a file named `model.lp`

In [2]:
#### gurobipy code
grid = m.addVars(time_periods, obj=1, name="grid")
m.write('model.lp')


**Q3.** [Depth of discharge](https://en.wikipedia.org/wiki/Depth_of_discharge) refers to the percentage a battery's level is below its capacity. To help maintain the batteries, a restriction is put into place that says the depth of discharge for **both batteries combined** cannot exceed 70% of the **total capacity**. Formulate constraints (write the inequalities algebraically) that model this and write code in gurobypi.

**Hints:** Make sure your generator expression is in parenthesis if you get an error and consider the initial state a few cells above.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Formulation:
$$
\sum_bs_{b,t} \ge 42 \quad \forall t \in T
$$



In [6]:
#### gurobipy code
state = m.addVars(batteries, time_periods, name="state")

discharge_constr = m.addConstrs((gp.quicksum(state[b,t] for b in batteries) >= 42 for t in time_periods), name='depth_of_discharge')

**Q4.** At the start of each day (other than the first) it is required that each battery be at least 40% full. The time periods for the start of each day are 30, 60, 90, 120, and 150. Formulate these constraints and write gurpbipy code. Do not worry about the solar forecast nor the contrstaint in **Q3**.

Formulation:
$$
s_{b,t} >= 0.4c_b \quad \forall b \in batteries \quad t \in T
$$

In [12]:
#### gurobipy code
m = gp.Model() # define the model again to make sure it is reset
state = m.addVars(batteries, time_periods, name="state")

minimum_state_constr = m.addConstrs(( state[b,t]>= 0.4*capacity[b] for b in batteries for t in range(30,180,30) ))

### Modeling with Binary Variables

You have five light bulbs that are either on or off. Let $y_i = 1$ if lighbulb $i$ is on and $y_i = 0$ if it is off, for$\space i \in \{1,2,3,4,5\}$. Model the following statements using binary variables and $\le,\space\ge,\space=$. All questions are independent of each other.

**Q4.** No more than three of the lightbulbs can be on.

In [16]:
bulbs = m.addVars(range(1,6,1), vtype=GRB.BINARY, name='bulb_switches')
m.addConstr(( gp.quicksum(bulbs[i] for i in range(1,6,1)) <= 3 ))

<gurobi.Constr *Awaiting Model Update*>

**Q5.** Lightbulb 2 or ligthbulb 3 must be on.

In [17]:
m.addConstr(( bulbs[2] + bulbs[3] >= 1 ))

<gurobi.Constr *Awaiting Model Update*>

**Q6.** If lightbulb 1 is on, then lightbulb 5 is also on.

In [18]:
m.addConstr(( bulbs[1] <= bulbs[5] ))

<gurobi.Constr *Awaiting Model Update*>

**Q7.** If lightbulb 1 is off, then lightbulb 3 is on.

In [20]:
m.addConstr(( (1 - bulbs[1])*(1 - bulbs[3]) == 0 ))

<gurobi.QConstr *Awaiting Model Update*>

**Q8.** If lightbulb 1 is off, then lightbulb 2 is off

In [ ]:
m.addConstr(( (1 - bulbs[1])*bulbs[2] == 0 ))

**Q9.** Lightbulb 2 and lightbulb 4 are in opposite on/off states.

In [ ]:
m.addConstr(( bulbs[2] + bulbs[4] == 1 ))

**Q10.** If any of lightbulbs 1 through 4 are on, then lighbtulb 5 is also on.

In [22]:
m.addConstr(( gp.quicksum(bulbs[i] for i in range(1,5,1))*bulbs[5] >= 1 ))

<gurobi.QConstr *Awaiting Model Update*>

In [23]:
m.write('model2.lp')